In [0]:
-- Create table from sample dataset
CREATE OR REPLACE TABLE lakehouse.raw.nyc_taxi_trips
USING DELTA
AS SELECT * FROM samples.nyctaxi.trips
LIMIT 100000;

-- Verify the data
SELECT COUNT(*) as total_records FROM lakehouse.raw.nyc_taxi_trips;

In [0]:
%python

dbutils.fs.ls('/databricks-datasets')
dbutils.fs.ls('dbfs:/')

dbutils.widgets.text("sample_file_path", "", "File path to load")

In [0]:
%python
from pyspark.sql.types import *

# Define source dataset path (example using Databricks sample dataset)
source_path = dbutils.widgets.get("sample_file_path")

# Define target table in lakehouse.raw schema
target_table = "lakehouse.raw.iot_device_data"

# Define schema for the data (optional, for explicit control)
schema = StructType([
    StructField("device_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True),
    StructField("pressure", DoubleType(), True)
])

# Read data from source
df = spark.read.schema(schema).json(source_path)

# Write data to lakehouse.raw schema as Delta table
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(target_table)

print(f"Data successfully loaded into {target_table}")